# LeetCode 2326 — Spiral Matrix IV bằng Python

Link đề: https://leetcode.com/problems/spiral-matrix-iv/ · Độ khó: **Medium** · Chủ đề: Array, Linked List, Matrix, Simulation

Skeleton LeetCode:

```python
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def spiralMatrix(self, m: int, n: int, head: Optional[ListNode]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

1. Tự dựng lớp `ListNode` và các hàm **dựng / xuất** danh sách liên kết để chạy thử ngoài LeetCode.
2. Mô phỏng đường đi **xoắn ốc theo chiều kim đồng hồ** bằng hai cách: (a) **quay hướng** khi gặp biên hoặc ô đã điền; (b) **bóc từng lớp** với bốn biên `top, bottom, left, right`.
3. Duyệt danh sách liên kết **đồng thời** với đường xoắn ốc và dừng đúng lúc danh sách hết (phần còn lại giữ `−1`).
4. Tránh các bẫy kinh điển: `[[-1] * n] * m` (các hàng **dùng chung** một list), lặp lại hàng/cột giữa khi lớp cuối chỉ còn một hàng hoặc một cột, quay sai chiều.
5. Liên hệ họ bài xoắn ốc: 54 (đọc), 59 (điền 1…n²), 885, và biến thể ngược chiều kim đồng hồ.

## 2. Đề bài & ví dụ

Cho hai số `m`, `n` (kích thước ma trận) và `head` của một danh sách liên kết các số nguyên. Tạo ma trận `m × n` chứa các giá trị của danh sách theo thứ tự **xoắn ốc chiều kim đồng hồ**, bắt đầu từ góc trên-trái. Ô nào còn trống thì điền `−1`.

**Ví dụ 1:** `m = 3`, `n = 5`, `head = [3,0,2,6,8,1,7,9,4,2,5,5,0]` → `[[3,0,2,6,8],[5,0,-1,-1,1],[5,2,4,9,7]]`.

**Ví dụ 2:** `m = 1`, `n = 4`, `head = [0,1,2]` → `[[0,1,2,-1]]`.

**Ràng buộc:** `1 ≤ m, n ≤ 10⁵`; `1 ≤ m·n ≤ 10⁵`; số nút trong `[1, m·n]`; `0 ≤ Node.val ≤ 1000`.

In [1]:
from typing import List, Optional
import random, time

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
    def __repr__(self):
        return f"ListNode({self.val})"

def build_list(vals):
    dummy = ListNode(); cur = dummy
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

def to_list(head):
    out = []
    while head:
        out.append(head.val); head = head.next
    return out

examples = [(3, 5, [3, 0, 2, 6, 8, 1, 7, 9, 4, 2, 5, 5, 0], [[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]]),
            (1, 4, [0, 1, 2], [[0, 1, 2, -1]])]
h = build_list(examples[0][2]); print("dựng rồi xuất lại:", to_list(h), "| nút đầu:", h, "| nút thứ hai:", h.next)

dựng rồi xuất lại: [3, 0, 2, 6, 8, 1, 7, 9, 4, 2, 5, 5, 0] | nút đầu: ListNode(3) | nút thứ hai: ListNode(0)


## 3. Kiến thức nền

### 3.1 Duyệt danh sách liên kết

Không có truy cập ngẫu nhiên: chỉ đi `head = head.next` từng bước; `None` báo hết danh sách. Mỗi giá trị chỉ được đọc **một lần** theo thứ tự.

### 3.2 Tạo ma trận đúng cách

In [2]:
bad = [[-1] * 3] * 2; bad[0][0] = 9
good = [[-1] * 3 for _ in range(2)]; good[0][0] = 9
print("[[-1]*3]*2 sau khi sửa bad[0][0]:", bad, "<- hai hàng là CÙNG một list")
print("list comprehension            :", good)

[[-1]*3]*2 sau khi sửa bad[0][0]: [[9, -1, -1], [9, -1, -1]] <- hai hàng là CÙNG một list
list comprehension            : [[9, -1, -1], [-1, -1, -1]]


### 3.3 Bốn hướng theo chiều kim đồng hồ

Phải `(0, +1)` → xuống `(+1, 0)` → trái `(0, −1)` → lên `(−1, 0)` → phải… Mẹo viết gọn: `dirs = (0, 1, 0, −1, 0)`, hướng `k` là `(dirs[k], dirs[k+1])`.

In [3]:
dirs = (0, 1, 0, -1, 0)
print([(dirs[k], dirs[k + 1]) for k in range(4)], "= phải, xuống, trái, lên")

[(0, 1), (1, 0), (0, -1), (-1, 0)] = phải, xuống, trái, lên


### 3.4 Thứ tự xoắn ốc của các ô

Đi theo hướng hiện tại; nếu ô tiếp theo ra ngoài biên hoặc đã đi qua thì **rẽ phải** (k += 1). Thứ tự này phủ mọi ô đúng một lần.

In [4]:
def spiral_order(m, n):
    seen = [[False] * n for _ in range(m)]; out = []; i = j = k = 0
    for _ in range(m * n):
        out.append((i, j)); seen[i][j] = True
        x, y = i + dirs[k], j + dirs[k + 1]
        if not (0 <= x < m and 0 <= y < n) or seen[x][y]:
            k = (k + 1) % 4; x, y = i + dirs[k], j + dirs[k + 1]
        i, j = x, y
    return out

print(spiral_order(3, 4))

[(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (2, 2), (2, 1), (2, 0), (1, 0), (1, 1), (1, 2)]


## 4. Ý tưởng nền tảng / trực giác

Đi theo thứ tự xoắn ốc các ô; lần lượt đặt giá trị của nút hiện tại vào ô hiện tại rồi sang nút kế tiếp. Khi danh sách hết thì dừng — các ô chưa đi tới vẫn giữ `−1` (đã khởi tạo sẵn).

Ví dụ 1 (`3 × 5`, 13 giá trị, 15 ô): thứ tự ô là hàng 0 trái→phải (5 ô: `3 0 2 6 8`), cột 4 xuống (2 ô: `1 7`), hàng 2 phải→trái (4 ô: `9 4 2 5`), cột 0 lên (1 ô: `5`), rồi hàng 1 trái→phải (`0`, rồi hết danh sách) ⇒ hai ô `(1, 2)`, `(1, 3)` giữ `−1`.

In [5]:
m, n, vals, exp = examples[0]
order = spiral_order(m, n)
grid = [["  ."] * n for _ in range(m)]
for step, (i, j) in enumerate(order):
    grid[i][j] = f"{step:>3}"
print("thứ tự đi qua các ô:"); [print(" ".join(r)) for r in grid]
print("13 giá trị lấp 13 ô đầu (bước 0..12); ô bước 13, 14 giữ -1")

thứ tự đi qua các ô:
  0   1   2   3   4
 11  12  13  14   5
 10   9   8   7   6
13 giá trị lấp 13 ô đầu (bước 0..12); ô bước 13, 14 giữ -1


## 5. Cách 1 — Brute Force / baseline

### 5.1 Chuyển danh sách sang mảng, sinh thứ tự ô bằng `spiral_order`, rồi điền — O(m·n)

In [6]:
def brute(m, n, head):
    vals = to_list(head)
    ans = [[-1] * n for _ in range(m)]
    for (i, j), v in zip(spiral_order(m, n), vals):
        ans[i][j] = v
    return ans

for m, n, vals, exp in examples:
    got = brute(m, n, build_list(vals)); print(got, "OK" if got == exp else "SAI")

[[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]] OK
[[0, 1, 2, -1]] OK


Đây đã là O(m·n) — "brute" ở đây nghĩa là cách **tách bước** rõ ràng nhất (sinh thứ tự ô riêng, dùng mảng `seen` riêng, chép danh sách ra mảng), tốn thêm bộ nhớ O(m·n) cho `seen`, thứ tự ô và mảng giá trị.

### 5.2 Baseline 2 — Đi theo xoắn ốc bằng cách "thử mọi hướng" ở mỗi bước

Mỗi bước thử lần lượt 4 hướng bắt đầu từ hướng hiện tại, chọn hướng đầu tiên hợp lệ. Đúng, nhưng thử thừa.

In [7]:
def try_all_dirs(m, n, head):
    ans = [[-1] * n for _ in range(m)]; seen = [[False] * n for _ in range(m)]
    i = j = k = 0
    while head:
        ans[i][j] = head.val; seen[i][j] = True; head = head.next
        if not head: break
        for t in range(4):
            kk = (k + t) % 4; x, y = i + dirs[kk], j + dirs[kk + 1]
            if 0 <= x < m and 0 <= y < n and not seen[x][y]:
                i, j, k = x, y, kk; break
    return ans

for m, n, vals, exp in examples:
    got = try_all_dirs(m, n, build_list(vals)); print(got, "OK" if got == exp else "SAI")

[[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]] OK
[[0, 1, 2, -1]] OK


**Nút thắt:** không có nút thắt về độ phức tạp (mọi cách đều O(m·n)), chỉ là bộ nhớ phụ và sự gọn gàng: ta có thể dùng chính giá trị `−1` của ma trận làm dấu "chưa điền" (vì `Node.val ≥ 0`), và chỉ cần rẽ **tối đa một lần** mỗi bước.

## 6. Ý tưởng thuật toán / bước đột phá

**Cách A — quay hướng (doocs):** ma trận khởi tạo `−1`; ở mỗi bước điền ô hiện tại, sang nút kế; nếu ô phía trước ra biên hoặc **đã khác `−1`** thì rẽ phải. Dùng `−1` làm "chưa thăm" hợp lệ vì `0 ≤ val`.

**Cách B — bóc lớp:** giữ `top, bottom, left, right`. Mỗi lớp: hàng `top` trái→phải, cột `right` trên→dưới, hàng `bottom` phải→trái (nếu `top < bottom`), cột `left` dưới→trên (nếu `left < right`). Thu hẹp bốn biên, lặp tới khi hết danh sách.

Cả hai O(m·n) thời gian, O(1) bộ nhớ phụ (ngoài ma trận kết quả).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quay hướng, dùng `−1` làm dấu chưa thăm (cách doocs)

In [8]:
def turn_simulation(m, n, head):
    ans = [[-1] * n for _ in range(m)]
    i = j = k = 0
    while True:
        ans[i][j] = head.val
        head = head.next
        if head is None: break
        while True:
            x, y = i + dirs[k], j + dirs[k + 1]
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                i, j = x, y; break
            k = (k + 1) % 4
    return ans

for m, n, vals, exp in examples:
    got = turn_simulation(m, n, build_list(vals)); print(got, "OK" if got == exp else "SAI")

[[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]] OK
[[0, 1, 2, -1]] OK


Vòng `while True` bên trong rẽ tối đa một lần trước khi tìm được ô hợp lệ — vì danh sách còn nút ⇒ còn ô trống, và ô trống kế tiếp theo xoắn ốc luôn nằm ở hướng hiện tại hoặc hướng rẽ phải.

### 7.2 Cách 3 — Bóc từng lớp với bốn biên

In [9]:
def layers(m, n, head):
    ans = [[-1] * n for _ in range(m)]
    top, bottom, left, right = 0, m - 1, 0, n - 1
    def put(i, j):
        nonlocal head
        if head is None: return False
        ans[i][j] = head.val; head = head.next
        return True
    while head and top <= bottom and left <= right:
        for j in range(left, right + 1):
            if not put(top, j): return ans
        for i in range(top + 1, bottom + 1):
            if not put(i, right): return ans
        if top < bottom:
            for j in range(right - 1, left - 1, -1):
                if not put(bottom, j): return ans
        if left < right:
            for i in range(bottom - 1, top, -1):
                if not put(i, left): return ans
        top, bottom, left, right = top + 1, bottom - 1, left + 1, right - 1
    return ans

for m, n, vals, exp in examples:
    got = layers(m, n, build_list(vals)); print(got, "OK" if got == exp else "SAI")

[[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]] OK
[[0, 1, 2, -1]] OK


Hai điều kiện `top < bottom` và `left < right` ngăn đi lại một hàng/cột khi lớp trong cùng chỉ là **một hàng** hoặc **một cột** (Lỗi 3).

### 7.3 Cách 4 — Đi theo "độ dài đoạn": `n, m−1, n−1, m−2, n−2, …`

Đường xoắn ốc gồm các đoạn thẳng có độ dài giảm dần xen kẽ theo chiều ngang/dọc. Không cần kiểm tra biên hay ô đã thăm.

In [10]:
def segment_lengths(m, n, head):
    ans = [[-1] * n for _ in range(m)]
    i, j, k = 0, -1, 0                                # bắt đầu "trước" ô (0, 0)
    lens = [n, m - 1]                                 # độ dài đoạn ngang / dọc kế tiếp
    while head:
        L = lens[k % 2]
        if L <= 0: break
        for _ in range(L):
            i, j = i + dirs[k], j + dirs[k + 1]
            ans[i][j] = head.val; head = head.next
            if head is None: return ans
        lens[k % 2] -= 1                              # đoạn cùng loại tiếp theo ngắn hơn 1
        k = (k + 1) % 4
    return ans

for m, n, vals, exp in examples:
    got = segment_lengths(m, n, build_list(vals)); print(got, "OK" if got == exp else "SAI")

[[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]] OK
[[0, 1, 2, -1]] OK


Độ dài các đoạn: ngang `n`, dọc `m − 1`, ngang `n − 1`, dọc `m − 2`, … — mỗi lần rẽ, đoạn **cùng loại** kế tiếp ngắn hơn 1 vì một hàng/cột vừa được lấp xong. Tổng các độ dài đúng bằng `m·n` và quá trình dừng khi một độ dài về 0 — nên cách này tự xử lý được trường hợp lớp cuối là một hàng/cột mà không cần điều kiện đặc biệt nào.

## 8. TRACE từng bước

### 8.1 Trace cách 2 trên ví dụ 2 và một ma trận vuông nhỏ

In [11]:
def trace_turn(m, n, vals):
    ans = [[-1] * n for _ in range(m)]; head = build_list(vals); i = j = k = 0; names = "→↓←↑"
    while True:
        ans[i][j] = head.val; print(f"  đặt {head.val} vào ({i},{j}), hướng {names[k]}")
        head = head.next
        if head is None: break
        turned = 0
        while True:
            x, y = i + dirs[k], j + dirs[k + 1]
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1: i, j = x, y; break
            k = (k + 1) % 4; turned += 1
        if turned: print(f"    rẽ phải -> hướng {names[k]}")
    return ans

print(trace_turn(1, 4, [0, 1, 2]))
print(trace_turn(3, 3, [1, 2, 3, 4, 5, 6, 7, 8, 9]))

  đặt 0 vào (0,0), hướng →
  đặt 1 vào (0,1), hướng →
  đặt 2 vào (0,2), hướng →
[[0, 1, 2, -1]]
  đặt 1 vào (0,0), hướng →
  đặt 2 vào (0,1), hướng →
  đặt 3 vào (0,2), hướng →
    rẽ phải -> hướng ↓
  đặt 4 vào (1,2), hướng ↓
  đặt 5 vào (2,2), hướng ↓
    rẽ phải -> hướng ←
  đặt 6 vào (2,1), hướng ←
  đặt 7 vào (2,0), hướng ←
    rẽ phải -> hướng ↑
  đặt 8 vào (1,0), hướng ↑
    rẽ phải -> hướng →
  đặt 9 vào (1,1), hướng →
[[1, 2, 3], [8, 9, 4], [7, 6, 5]]


### 8.2 Trace cách 4: các đoạn và độ dài

In [12]:
m, n = 3, 5; lens = [n, m - 1]; k = 0; total = 0; segs = []
while total < m * n and lens[k % 2] > 0:
    L = lens[k % 2]; segs.append(("→↓←↑"[k], L)); total += L; lens[k % 2] -= 1; k = (k + 1) % 4
print(segs, "tổng", total, "= m*n =", m * n)

[('→', 5), ('↓', 2), ('←', 4), ('↑', 1), ('→', 3)] tổng 15 = m*n = 15


## 9. Code nộp LeetCode

In [13]:
from typing import List, Optional

class Solution:
    def spiralMatrix(self, m: int, n: int, head: Optional[ListNode]) -> List[List[int]]:
        ans = [[-1] * n for _ in range(m)]          # mỗi hàng là một list riêng
        dirs = (0, 1, 0, -1, 0)                       # phải, xuống, trái, lên
        i = j = k = 0
        while head:
            ans[i][j] = head.val
            head = head.next
            x, y = i + dirs[k], j + dirs[k + 1]
            if not (0 <= x < m and 0 <= y < n and ans[x][y] == -1):
                k = (k + 1) % 4                       # gặp biên hoặc ô đã điền: rẽ phải
                x, y = i + dirs[k], j + dirs[k + 1]
            i, j = x, y
        return ans

In [14]:
sol = Solution()
for m, n, vals, exp in examples:
    got = sol.spiralMatrix(m, n, build_list(vals))
    print(got, "| OK" if got == exp else "| SAI")

[[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]] | OK
[[0, 1, 2, -1]] | OK


Bản nộp rẽ **tối đa một lần** mỗi bước (không cần vòng `while` bên trong). Sau khi điền ô cuối cùng của danh sách, `(i, j)` có thể trỏ tới một ô không hợp lệ (ra ngoài biên) nhưng vòng lặp đã dừng nên không truy cập.

## 10. Vì sao thuật toán đúng

**Bổ đề 1 (thứ tự xoắn ốc phủ mỗi ô đúng một lần).** Quy tắc "đi thẳng nếu ô phía trước trong biên và chưa thăm, ngược lại rẽ phải" sinh ra đường xoắn ốc từ ngoài vào trong: chạy hết hàng trên, cột phải, hàng dưới, cột trái của lớp ngoài, rồi lặp với lớp trong. Mỗi bước sang một ô **chưa thăm**, nên không ô nào được thăm hai lần.

**Bổ đề 2 (chỉ cần rẽ một lần).** Giả sử đang ở ô `c` và còn ô chưa thăm. Nếu ô phía trước không hợp lệ, ta đang ở **góc** của lớp hiện tại; ô kế tiếp theo xoắn ốc nằm ngay bên phải (theo hướng rẽ). Ô đó chưa thăm, vì nếu không thì `c` đã là ô cuối của xoắn ốc — mâu thuẫn với "còn ô chưa thăm". (Cách doocs dùng vòng `while` an toàn cho mọi trường hợp, bản nộp dùng một lần rẽ.)

**Bổ đề 3 (`−1` là dấu chưa thăm hợp lệ).** Ô đã điền chứa `val ∈ [0, 1000]`, khác `−1`; ô chưa điền là `−1` ⇒ `ans[x][y] == −1` ⇔ chưa thăm.

**Kết luận.** Nút thứ `t` (0-index) của danh sách được đặt vào ô thứ `t` theo thứ tự xoắn ốc; số nút ≤ `m·n` nên không vượt quá; các ô sau đó giữ `−1`. ∎

**Cách bóc lớp.** Mỗi lớp `(top, bottom, left, right)` điền đúng viền của hình chữ nhật con, không lặp ô nhờ hai điều kiện `top < bottom`, `left < right`; lớp tiếp theo là hình chữ nhật thu nhỏ 1 về mỗi phía.

## 11. Phân tích Time / Space Complexity

`m·n ≤ 10⁵`, `L` = số nút ≤ `m·n`.

| Cách | Time | Space phụ (ngoài kết quả) |
|---|---|---|
| 1. Tách bước | O(m·n) | O(m·n) (`seen`, thứ tự ô, mảng giá trị) |
| 1b. Thử mọi hướng | O(m·n) | O(m·n) (`seen`) |
| 2. Quay hướng (doocs) | O(m·n) khởi tạo + O(L) | O(1) |
| 3. Bóc lớp | O(m·n) + O(L) | O(1) |
| 4. Độ dài đoạn | O(m·n) + O(L) | O(1) |

Khởi tạo ma trận `−1` luôn tốn O(m·n); phần điền tốn O(L).

In [15]:
rng = random.Random(2326)
for m, n in [(316, 316), (1, 10**5), (10**5, 1), (250, 400)]:
    vals = [rng.randint(0, 1000) for _ in range(rng.randint(1, m * n))]
    res = []
    for name, f in [("brute", brute), ("turn_simulation", turn_simulation), ("layers", layers),
                    ("segment_lengths", segment_lengths), ("Solution", Solution().spiralMatrix)]:
        head = build_list(vals)
        t0 = time.perf_counter(); res.append(f(m, n, head)); dt = time.perf_counter() - t0
        print(f"{m}x{n:<6} {name:<16} {dt*1000:7.1f} ms")
    assert all(r == res[0] for r in res)

316x316    brute               31.7 ms


316x316    turn_simulation     11.5 ms


316x316    layers               5.3 ms


316x316    segment_lengths      6.0 ms


316x316    Solution            11.1 ms


1x100000 brute               42.3 ms


... (4 outputs left out)

100000x1      segment_lengths     44.9 ms
100000x1      Solution            16.2 ms


250x400    brute               53.2 ms


250x400    turn_simulation     13.7 ms
250x400    layers               5.9 ms


250x400    segment_lengths      6.3 ms
250x400    Solution            13.2 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [16]:
ALL = {"brute": brute, "try_all_dirs": try_all_dirs, "turn_simulation": turn_simulation, "layers": layers,
       "segment_lengths": segment_lengths, "Solution": Solution().spiralMatrix}
edge = examples + [
    (1, 1, [7], [[7]]),
    (3, 1, [1, 2], [[1], [2], [-1]]),                         # một cột
    (2, 2, [1, 2, 3, 4], [[1, 2], [4, 3]]),
    (3, 3, [1, 2, 3, 4, 5, 6, 7, 8, 9], [[1, 2, 3], [8, 9, 4], [7, 6, 5]]),
    (2, 3, [1], [[1, -1, -1], [-1, -1, -1]]),
    (4, 3, list(range(12)), [[0, 1, 2], [9, 10, 3], [8, 11, 4], [7, 6, 5]]),   # lớp trong là một cột
]
for m, n, vals, exp in edge:
    for name, f in ALL.items():
        got = f(m, n, build_list(vals))
        assert got == exp, (name, m, n, vals, got)
    print(f"{m}x{n} {vals} -> {exp}")
print("edge OK")

3x5 [3, 0, 2, 6, 8, 1, 7, 9, 4, 2, 5, 5, 0] -> [[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]]
1x4 [0, 1, 2] -> [[0, 1, 2, -1]]
1x1 [7] -> [[7]]
3x1 [1, 2] -> [[1], [2], [-1]]
2x2 [1, 2, 3, 4] -> [[1, 2], [4, 3]]
3x3 [1, 2, 3, 4, 5, 6, 7, 8, 9] -> [[1, 2, 3], [8, 9, 4], [7, 6, 5]]
2x3 [1] -> [[1, -1, -1], [-1, -1, -1]]
4x3 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] -> [[0, 1, 2], [9, 10, 3], [8, 11, 4], [7, 6, 5]]
edge OK


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

In [17]:
rng = random.Random(42)
for _ in range(1500):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    vals = [rng.randint(0, 1000) for _ in range(rng.randint(1, m * n))]
    ref = brute(m, n, build_list(vals))
    for name, f in ALL.items():
        assert f(m, n, build_list(vals)) == ref, (name, m, n, vals)
print("1500 bộ ngẫu nhiên OK")

1500 bộ ngẫu nhiên OK


### 12.3 Tính chất: đọc lại ma trận theo xoắn ốc (LC 54) cho đúng danh sách rồi toàn `−1`

In [18]:
def read_spiral(mat):
    m, n = len(mat), len(mat[0])
    return [mat[i][j] for i, j in spiral_order(m, n)]

rng = random.Random(7)
for _ in range(1000):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    vals = [rng.randint(0, 1000) for _ in range(rng.randint(1, m * n))]
    out = Solution().spiralMatrix(m, n, build_list(vals))
    assert len(out) == m and all(len(r) == n for r in out)
    assert read_spiral(out) == vals + [-1] * (m * n - len(vals))
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: `1 × 10⁵`, `10⁵ × 1`, và danh sách đúng `m·n` nút

In [19]:
vals = list(range(10**5))
assert Solution().spiralMatrix(1, 10**5, build_list(vals)) == [vals]
assert Solution().spiralMatrix(10**5, 1, build_list(vals)) == [[v] for v in vals]
out = Solution().spiralMatrix(316, 316, build_list([1] * (316 * 316)))
assert all(v == 1 for r in out for v in r)
print("cực trị OK")

cực trị OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tạo ma trận bằng `[[-1] * n] * m` (các hàng dùng chung)

In [20]:
def wrong_alias(m, n, head):
    ans = [[-1] * n] * m                                        # SAI: m tham chiếu tới CÙNG một hàng
    i = j = k = 0
    while head:
        ans[i][j] = head.val; head = head.next
        x, y = i + dirs[k], j + dirs[k + 1]
        if not (0 <= x < m and 0 <= y < n and ans[x][y] == -1):
            k = (k + 1) % 4; x, y = i + dirs[k], j + dirs[k + 1]
        i, j = x, y
    return [r[:] for r in ans]

for m, n, vals, exp in examples:
    print("bản sai :", wrong_alias(m, n, build_list(vals)), "\nbản đúng:", exp)

bản sai : [[3, 0, 2, 5, 0], [3, 0, 2, 5, 0], [3, 0, 2, 5, 0]] 
bản đúng: [[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]]
bản sai : [[0, 1, 2, -1]] 
bản đúng: [[0, 1, 2, -1]]


**Vì sao sai:** ví dụ 2 (`m = 1`) **trùng hợp** đúng. Với `m ≥ 2`, ghi vào hàng 0 cũng là ghi vào mọi hàng ⇒ dấu "chưa thăm" sai và đường đi lộn xộn. **Sửa:** `[[-1] * n for _ in range(m)]`.

### Lỗi 2 — Không kiểm tra danh sách hết (truy cập `None.val`)

In [21]:
def wrong_no_stop(m, n, head):
    ans = [[-1] * n for _ in range(m)]
    try:
        for i, j in spiral_order(m, n):
            ans[i][j] = head.val; head = head.next              # SAI: head có thể là None
        return ans
    except AttributeError as e:
        return f"AttributeError: {e}"

for m, n, vals, exp in examples:
    print("bản sai :", wrong_no_stop(m, n, build_list(vals)), "| bản đúng:", exp)

bản sai : AttributeError: 'NoneType' object has no attribute 'val' | bản đúng: [[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]]
bản sai : AttributeError: 'NoneType' object has no attribute 'val' | bản đúng: [[0, 1, 2, -1]]


**Sửa:** dừng khi `head is None` (hoặc dùng `zip` với danh sách giá trị như bản brute).

### Lỗi 3 — Bóc lớp không có điều kiện `top < bottom` / `left < right` (đi lại hàng/cột giữa)

In [22]:
def wrong_layers(m, n, head):
    ans = [[-1] * n for _ in range(m)]
    top, bottom, left, right = 0, m - 1, 0, n - 1
    cells = []
    while top <= bottom and left <= right:
        cells += [(top, j) for j in range(left, right + 1)]
        cells += [(i, right) for i in range(top + 1, bottom + 1)]
        cells += [(bottom, j) for j in range(right - 1, left - 1, -1)]   # SAI: thiếu if top < bottom
        cells += [(i, left) for i in range(bottom - 1, top, -1)]          # SAI: thiếu if left < right
        top, bottom, left, right = top + 1, bottom - 1, left + 1, right - 1
    vals = to_list(head)
    for t, (i, j) in enumerate(cells):                    # điền MỌI ô trong danh sách ô (giá trị hoặc -1)
        ans[i][j] = vals[t] if t < len(vals) else -1
    return ans

for m, n, vals, exp in examples:
    print("bản sai :", wrong_layers(m, n, build_list(vals)), "\nbản đúng:", exp)

bản sai : [[3, 0, 2, 6, 8], [5, -1, -1, -1, 1], [5, 2, 4, 9, 7]] 
bản đúng: [[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]]
bản sai : [[-1, -1, -1, -1]] 
bản đúng: [[0, 1, 2, -1]]


**Vì sao sai:** thiếu hai điều kiện, lớp cuối chỉ có một hàng (ví dụ 1: hàng 1, cột 1…3) bị đi **hai lần**: trái→phải rồi phải→trái. Danh sách ô có `m·n + 2` phần tử, hai ô cuối là `(1, 2)` và `(1, 1)` — được điền lại `−1`, xóa mất giá trị `0` ở `(1, 1)`. Ví dụ 2 cũng là một hàng: ba ô `(0, 2), (0, 1), (0, 0)` bị ghi đè `−1`. (Nếu chỉ điền khi còn giá trị thì lỗi bị **che** — các ô trùng luôn nằm cuối danh sách ô — nhưng mọi cách cài đặt "điền cả `−1`" hoặc "đếm số ô" đều lộ ra.) **Sửa:** thêm `if top < bottom` và `if left < right`.

### Lỗi 4 — Quay **ngược** chiều kim đồng hồ (thứ tự hướng sai)

In [23]:
def wrong_ccw(m, n, head):
    d = (1, 0, -1, 0, 1)                                         # SAI: xuống, phải, lên, trái
    ans = [[-1] * n for _ in range(m)]; i = j = k = 0
    while head:
        ans[i][j] = head.val; head = head.next
        x, y = i + d[k], j + d[k + 1]
        if not (0 <= x < m and 0 <= y < n and ans[x][y] == -1):
            k = (k + 1) % 4; x, y = i + d[k], j + d[k + 1]
        i, j = x, y
    return ans

for m, n, vals, exp in examples:
    print("bản sai :", wrong_ccw(m, n, build_list(vals)), "\nbản đúng:", exp)

bản sai : [[3, -1, -1, -1, -1], [1, 0, 9, 4, 8], [2, 5, 5, 2, 6]] 
bản đúng: [[3, 0, 2, 6, 8], [5, 0, -1, -1, 1], [5, 2, 4, 9, 7]]
bản sai : [[0, -1, -1, 2]] 
bản đúng: [[0, 1, 2, -1]]


**Vì sao sai:** đề yêu cầu chiều kim đồng hồ bắt đầu sang **phải**. Ma trận một hàng thì vẫn ra đúng (rẽ ngay sang phải) — ví dụ 2 **trùng hợp** đúng.

### Lỗi 5 — Dùng `ans[x][y] != -1` làm dấu "đã thăm" khi giá trị có thể là `−1`

Trong bài gốc `val ≥ 0` nên an toàn; nhưng nếu biến thể cho phép `val = −1`, ô đã điền `−1` bị coi là chưa thăm:

In [24]:
def turn_with_guard(m, n, vals):
    ans = [[-1] * n for _ in range(m)]; i = j = k = 0; t = 0
    while t < len(vals):
        ans[i][j] = vals[t]; t += 1
        if t == len(vals): break
        for _ in range(5):
            x, y = i + dirs[k], j + dirs[k + 1]
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1: i, j = x, y; break
            k = (k + 1) % 4
        else:
            return "KẸT: không tìm được ô 'trống'"
    return ans

m, n, vals = 3, 4, [-1, 1, -1, 1, 2, 2, 2, 2, 1, 2, -1, -1]
print("dùng -1 làm dấu :", turn_with_guard(m, n, vals))
print("brute (có seen) :", brute(m, n, build_list(vals)))
print("[1, -1, 2, 3] trong 2x2:", turn_with_guard(2, 2, [1, -1, 2, 3]), "(trùng hợp đúng)")

dùng -1 làm dấu : KẸT: không tìm được ô 'trống'
brute (có seen) : [[-1, 1, -1, 1], [2, -1, -1, 2], [1, 2, 2, 2]]
[1, -1, 2, 3] trong 2x2: [[1, -1], [3, 2]] (trùng hợp đúng)


**Vì sao sai:** ô đã điền giá trị `−1` bị coi là "chưa thăm", đường đi quay lại ô đó; ở ví dụ `3 × 4` trên, sau vài bước thuật toán thậm chí không còn hướng nào hợp lệ (bản gốc không có giới hạn vòng lặp sẽ **lặp vô hạn**). Ví dụ `2 × 2` nhỏ thì trùng hợp đúng.

**Sửa (cho biến thể):** dùng mảng `seen` riêng (cách 1b) hoặc cách bóc lớp / độ dài đoạn — hai cách này không cần dấu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Xoắn ốc ⇒ hoặc **quay hướng** khi bị chặn, hoặc **bóc lớp** bốn biên, hoặc **độ dài đoạn** giảm dần.
- `dirs = (0, 1, 0, −1, 0)` cho bốn hướng chiều kim đồng hồ.
- Ma trận 2D trong Python: luôn dùng list comprehension.
- Danh sách liên kết: dừng khi `None`; không đếm trước độ dài nếu không cần.
- Kiểm thử bằng cách **đọc lại** theo xoắn ốc (bài 54) — phép nghịch đảo của bài 59/2326.
- Test phải gồm ma trận **một hàng**, **một cột**, và ma trận có lớp trong cùng là một hàng/cột (ví dụ `3 × 5`, `4 × 3`) — đó là chỗ các cài đặt bóc lớp hay sai. Đồng thời thử danh sách **đủ** `m·n` nút để lộ các lỗi bị che khi danh sách ngắn.
- Bài cùng họ: 54 Spiral Matrix, 59 Spiral Matrix II, 885 Spiral Matrix III, 2326, 1914 Cyclically Rotating a Grid, 48 Rotate Image.

## 15. Biến thể của bài

### 15.1 Biến thể A — Ngược chiều kim đồng hồ (bắt đầu đi **xuống**)

In [25]:
def spiral_ccw(m, n, head):
    return wrong_ccw(m, n, head)                                  # Lỗi 4 là lời giải đúng của biến thể này

print(spiral_ccw(3, 3, build_list(range(1, 10))))

[[1, -1, -1], [6, 7, 5], [9, 8, 4]]


### 15.2 Biến thể B — LC 59: điền `1 … n²` vào ma trận vuông (không có danh sách)

In [26]:
def generate_matrix(n):
    return Solution().spiralMatrix(n, n, build_list(range(1, n * n + 1)))
for row in generate_matrix(4): print(row)

[1, 2, 3, 4]
[12, 13, 14, 5]
[11, 16, 15, 6]
[10, 9, 8, 7]


### 15.3 Biến thể C — Giá trị có thể âm (kể cả `−1`): dùng cách độ dài đoạn (không cần dấu)

In [27]:
rng = random.Random(153)
for _ in range(500):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    vals = [rng.randint(-3, 3) for _ in range(rng.randint(1, m * n))]
    assert segment_lengths(m, n, build_list(vals)) == brute(m, n, build_list(vals)) == layers(m, n, build_list(vals))
print(segment_lengths(2, 2, build_list([1, -1, 2, 3])))

[[1, -1], [3, 2]]


### 15.4 Biến thể D — Trả kết quả cũng dưới dạng **danh sách liên kết** đọc theo hàng

In [28]:
def matrix_to_linked_rows(mat):
    return build_list([v for row in mat for v in row])
print(to_list(matrix_to_linked_rows(Solution().spiralMatrix(3, 5, build_list(examples[0][2])))))

[3, 0, 2, 6, 8, 5, 0, -1, -1, 1, 5, 2, 4, 9, 7]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space phụ | Ưu / nhược |
|---|---|---|---|---|
| 1. Tách bước | thứ tự ô riêng | O(m·n) | O(m·n) | rõ ràng, tốn bộ nhớ |
| 1b. Thử mọi hướng | `seen` riêng | O(m·n) | O(m·n) | an toàn với mọi giá trị |
| 2. Quay hướng | `−1` làm dấu | O(m·n) | O(1) | **mặc định**, ngắn |
| 3. Bóc lớp | bốn biên | O(m·n) | O(1) | dễ sai biên lớp cuối |
| 4. Độ dài đoạn | `n, m−1, n−1, …` | O(m·n) | O(1) | không cần dấu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `spiralMatrix(2, 3, [1, 2, 3, 4, 5])` và `spiralMatrix(3, 2, [1, 2, 3, 4, 5, 6])`?

**Bài 2 (tính tay):** Liệt kê thứ tự ô xoắn ốc của ma trận `3 × 4`.

**Bài 3 (điền TODO):** Viết `bt3_todo(m, n, i, j)` trả về **chỉ số bước** (0-index) mà đường xoắn ốc đi qua ô `(i, j)`, O(1) hoặc O(số lớp).

**Bài 4 (tự viết lại):** Viết `bt4_todo(m, n, head)` cài cách **bóc lớp** không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(m, n, head)` điền xoắn ốc nhưng **bắt đầu từ góc dưới-phải**, đi sang **trái** trước, vẫn theo chiều kim đồng hồ.

In [29]:
def bt3_todo(m, n, i, j):
    # TODO: bước xoắn ốc của ô (i, j)
    pass

def bt4_todo(m, n, head):
    # TODO: bóc lớp
    pass

def bt5_todo(m, n, head):
    # TODO: bắt đầu từ góc dưới-phải
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `2 × 3`: hàng 0 `1 2 3`, cột 2 xuống `4`, hàng 1 sang trái `5`. Mức 2: `[[1,2,3],[−1,5,4]]`. Mức 3: `3 × 2`: `[[1,2],[6,3],[5,4]]`.

**Bài 2.** Mức 1: hàng 0: `(0,0)…(0,3)`. Mức 2: cột 3: `(1,3),(2,3)`; hàng 2: `(2,2),(2,1),(2,0)`; cột 0: `(1,0)`. Mức 3: lớp trong: `(1,1),(1,2)`.

**Bài 3.** Mức 1: lớp của ô `ℓ = min(i, j, m−1−i, n−1−j)`. Mức 2: số ô của các lớp ngoài = `m·n − (m − 2ℓ)(n − 2ℓ)`. Mức 3: trong lớp `ℓ`, xét ô nằm trên cạnh nào và cộng khoảng cách dọc theo viền.

**Bài 4.** Mức 1: bốn biên. Mức 2: bốn vòng `for` mỗi lớp. Mức 3: điều kiện `top < bottom`, `left < right`, dừng khi hết danh sách.

**Bài 5.** Mức 1: xoay ma trận 180°: góc dưới-phải ↔ góc trên-trái, hướng trái ↔ phải. Mức 2: giải bài gốc rồi xoay kết quả 180°. Mức 3: `[row[::-1] for row in ans[::-1]]`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[[1,2,3],[−1,5,4]]`, `[[1,2],[6,3],[5,4]]`. **Bài 2:** như gợi ý.

In [30]:
assert Solution().spiralMatrix(2, 3, build_list([1, 2, 3, 4, 5])) == [[1, 2, 3], [-1, 5, 4]]
assert Solution().spiralMatrix(3, 2, build_list([1, 2, 3, 4, 5, 6])) == [[1, 2], [6, 3], [5, 4]]
assert spiral_order(3, 4) == [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (2, 2), (2, 1), (2, 0), (1, 0), (1, 1), (1, 2)]

def bt3(m, n, i, j):
    l = min(i, j, m - 1 - i, n - 1 - j)
    before = m * n - (m - 2 * l) * (n - 2 * l)
    top, left, bottom, right = l, l, m - 1 - l, n - 1 - l
    w, h = right - left + 1, bottom - top + 1
    if i == top: return before + (j - left)
    if j == right: return before + w + (i - top - 1)
    if i == bottom: return before + w + (h - 1) + (right - 1 - j)
    return before + w + (h - 1) + (w - 1) + (bottom - 1 - i)

def bt4(m, n, head):
    return layers(m, n, head)

def bt5(m, n, head):
    ans = Solution().spiralMatrix(m, n, head)
    return [row[::-1] for row in ans[::-1]]

def bt5_brute(m, n, vals):
    order = [(m - 1 - i, n - 1 - j) for i, j in spiral_order(m, n)]
    ans = [[-1] * n for _ in range(m)]
    for (i, j), v in zip(order, vals): ans[i][j] = v
    return ans

rng = random.Random(1999)
for _ in range(500):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    order = spiral_order(m, n)
    assert all(bt3(m, n, i, j) == t for t, (i, j) in enumerate(order))
    vals = [rng.randint(0, 9) for _ in range(rng.randint(1, m * n))]
    assert bt4(m, n, build_list(vals)) == brute(m, n, build_list(vals))
    assert bt5(m, n, build_list(vals)) == bt5_brute(m, n, vals)
print(bt3(3, 5, 1, 2), bt5(2, 3, build_list([1, 2, 3, 4, 5])))
print("Đáp án bài tập: tất cả assert OK")

13 [[4, 5, -1], [3, 2, 1]]
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = [[-1] * n for _ in range(m)]        # KHÔNG dùng [[-1]*n]*m
dirs = (0, 1, 0, -1, 0)                    # phải, xuống, trái, lên
i = j = k = 0
while head:
    ans[i][j] = head.val; head = head.next
    x, y = i + dirs[k], j + dirs[k + 1]
    if not (0 <= x < m and 0 <= y < n and ans[x][y] == -1):
        k = (k + 1) % 4; x, y = i + dirs[k], j + dirs[k + 1]
    i, j = x, y
return ans
```

**Checklist:**
- [ ] Ma trận tạo bằng list comprehension.
- [ ] Dừng khi danh sách hết; ô còn lại giữ `−1`.
- [ ] Chiều kim đồng hồ, bắt đầu sang phải.
- [ ] Bóc lớp: điều kiện `top < bottom`, `left < right`.
- [ ] Tự dựng `ListNode`, `build_list`, `to_list` để chạy thử ngoài LeetCode.